In [1]:
# PROMPT REPAIR NOTEBOOK
# Goal: for each failure topic, generate prompt / policy improvements.

from pathlib import Path
import os
import json

import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env
load_dotenv()

# Create OpenAI client using the API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Project paths
PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data dir:", DATA_PROCESSED)


Project root: C:\Users\User\Documents\retailmind-prototype
Processed data dir: C:\Users\User\Documents\retailmind-prototype\data\processed


In [2]:
# Load per-topic summaries produced by topic_clustering.ipynb
summary_path = DATA_PROCESSED / "uss_english_topic_summaries.parquet"
df_topics_summary = pd.read_parquet(summary_path)

print("Topics summary rows:", len(df_topics_summary))
df_topics_summary.head()


Topics summary rows: 8


,topic_id,n_examples,top_issues,example_texts,example_reason
0,1,1143,"[MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT]","[I want to find a movie to wathc, can you sear...",The bot does not acknowledge the user's intent...
1,7,633,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, HANDOFF_...",[Can you check on another? I'm looking for som...,The bot fails to acknowledge the user's previo...
2,3,578,"[MISSING_CONTEXT, TONE_ISSUE, HANDOFF_REQUIRED]","[No. Don't do that., No I dont want to book an...",The bot fails to acknowledge the user's clear ...
3,2,298,"[MISSING_CONTEXT, WRONG_FACT, UNSUPPORTED_INTENT]","[No, I need it for 3 at 11:45 am., Evening 7:3...",The bot failed to acknowledge the user's previ...
4,6,275,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FACT]","[No, not just yet., No, I want 1 ticket to an ...",The bot fails to acknowledge the user's respon...


In [3]:
# Load row-level topics (low-satisfaction user turns with topic_id)
topics_path = DATA_PROCESSED / "uss_english_issue_topics.parquet"
df_topics = pd.read_parquet(topics_path)

print("Rows in topics dataset:", len(df_topics))
df_topics[["topic_id", "dataset", "conv_id", "turn_id", "text", "issues"]].head()


Rows in topics dataset: 3537


,topic_id,dataset,conv_id,turn_id,text,issues
0,5,SGD,0,11,"No, play it on bedroom speaker instead.",[MISSING_CONTEXT]
1,3,SGD,1,9,No. Don't do that.,[MISSING_CONTEXT]
2,4,SGD,1,11,Make a reservation at a nearby restaurant.,[UNSUPPORTED_INTENT]
3,2,SGD,1,17,"No, I need it for 3 at 11:45 am.",[MISSING_CONTEXT]
4,7,SGD,2,7,Can you check on another? I'm looking for some...,[MISSING_CONTEXT]


In [ ]:
from textwrap import indent
import numpy as np
import ast

def ensure_list(val):
    
    # Already a Python list
    if isinstance(val, list):
        return [str(x) for x in val]

    # NumPy array -> convert to list
    if isinstance(val, np.ndarray):
        return [str(x) for x in val.tolist()]

    # String that might represent a list literal
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return [str(x) for x in parsed]
        except Exception:
            # Fallback: treat as single string entry
            return [val]

    # Anything else -> empty list
    return []


def build_topic_description(summary_row, extra_examples=None) -> str:
    """
    Build a textual description of a topic to send to the LLM.
    """
    topic_id = summary_row["topic_id"]
    n_examples = summary_row["n_examples"]

    # Normalize top_issues
    top_issues = ensure_list(summary_row.get("top_issues", []))
    top_issues_str = ", ".join(top_issues) if top_issues else "[none]"

    # Normalize example_texts
    example_texts = ensure_list(summary_row.get("example_texts", []))

    # Add extra examples
    if extra_examples:
        example_texts.extend([str(x) for x in extra_examples])

    # Limit to 5
    example_texts = example_texts[:5]

    # Example reason
    example_reason = summary_row.get("example_reason", "")

    # Build description
    lines = []
    lines.append(f"Topic ID: {topic_id}")
    lines.append(f"Number of examples: {n_examples}")
    lines.append(f"Most frequent issue labels: {top_issues_str}")
    lines.append("")
    lines.append("Representative user messages:")

    if len(example_texts) > 0:
        for i, txt in enumerate(example_texts, start=1):
            lines.append(f"{i}) {txt}")
    else:
        lines.append("[No example texts available]")

    if isinstance(example_reason, np.ndarray):
        # Convert numpy array reason (rare)
        example_reason = " ".join(example_reason.tolist())

    if isinstance(example_reason, str) and example_reason.strip():
        lines.append("")
        lines.append("Diagnostic reason extracted from logs:")
        lines.append(example_reason)

    return "\n".join(lines)


desc_example = build_topic_description(df_topics_summary.iloc[0])
print(desc_example)



Topic ID: 1
Number of examples: 1143
Most frequent issue labels: MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT

Representative user messages:
1) I want to find a movie to wathc
2) can you search for movies shown in San Jose.i like Adventure movies.
3) I like Documentary movies. could you try again?

Diagnostic reason extracted from logs:
The bot does not acknowledge the user's intent to find a movie, which indicates a lack of context or understanding of the user's request.


In [7]:
def build_prompt_repair_instruction(topic_description: str) -> str:
    """
    Build the prompt we send to the LLM to ask for prompt/policy improvements.
    """
    return f"""
You are an expert conversation designer for virtual assistants.

You are given a TOPIC representing a cluster of low-satisfaction user turns.
The TOPIC description includes:
- the most frequent issue labels (from this taxonomy: MISSING_CONTEXT, WRONG_FACT, TONE_ISSUE, LOOP, UNSUPPORTED_INTENT, SLOW_RESPONSE, HANDOFF_REQUIRED, SUCCESS_BEST_PRACTICE),
- representative user messages,
- a diagnostic reason extracted from logs.

Your task:
1. Propose a short, human-readable "topic_label" (max 12 words).
2. Explain the most likely "root_cause" of this type of failure.
3. Suggest 3–5 concrete "suggested_prompt_changes": changes to the assistant's system prompt, conversation policies, or retrieval logic to reduce this failure.
4. Write a "system_prompt_snippet" that could be added to the assistant's system message to mitigate this topic.
5. Propose 2–4 "guardrail_rules": simple if-then rules or constraints that help avoid this failure.
6. Propose 1–3 "evaluation_checks": small behavioral tests we can run on sample conversations to verify the fix.

Output a JSON object with exactly this schema:
{{
  "topic_label": "string",
  "root_cause": "string",
  "suggested_prompt_changes": ["string", "..."],
  "system_prompt_snippet": "string",
  "guardrail_rules": ["string", "..."],
  "evaluation_checks": ["string", "..."]
}}

Be specific and practical. Focus on what developers or conversation designers can actually change.

Here is the TOPIC description:
---
{topic_description}
---
"""


In [9]:
def generate_prompt_repair(topic_description: str, model_name: str = "gpt-4o-mini") -> dict:
    """
    Call OpenAI to generate prompt repair suggestions for a topic.
    Returns a Python dict with the fields defined in the JSON schema.
    """
    prompt = build_prompt_repair_instruction(topic_description)

    response = client.chat.completions.create(
        model=model_name,
        temperature=0.3,  # some creativity but not too random
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "You are an expert conversation designer. Always output valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            },
        ],
    )

    content = response.choices[0].message.content
    data = json.loads(content)

    # Normalise fields and types
    data.setdefault("topic_label", "")
    data.setdefault("root_cause", "")
    data.setdefault("system_prompt_snippet", "")

    for key in ["suggested_prompt_changes", "guardrail_rules", "evaluation_checks"]:
        val = data.get(key, [])
        if isinstance(val, str):
            val = [val]
        elif not isinstance(val, list):
            val = [str(val)]
        data[key] = [str(x).strip() for x in val if str(x).strip()]

    return data



In [10]:
repairs = []

for _, row in df_topics_summary.iterrows():
    topic_id = row["topic_id"]

    # Optionally: gather a few extra example texts from df_topics for this topic
    extra_examples = (
        df_topics[df_topics["topic_id"] == topic_id]["text"]
        .head(3)
        .tolist()
    )

    topic_desc = build_topic_description(row, extra_examples=extra_examples)
    result = generate_prompt_repair(topic_desc)

    repairs.append({
        "topic_id": topic_id,
        **result
    })

df_repairs = pd.DataFrame(repairs)
df_repairs


,topic_id,topic_label,root_cause,suggested_prompt_changes,system_prompt_snippet,guardrail_rules,evaluation_checks
0,1,Movie Search Context and Tone Issues,The assistant fails to recognize and respond a...,[Enhance the assistant's ability to recognize ...,"When users express interest in movies, acknowl...","[If a user mentions a movie genre, respond wit...",[Test if the assistant can successfully identi...
1,7,Contextual Awareness and Intent Support Issues,The assistant lacks the ability to retain and ...,[Enhance context retention capabilities to tra...,Ensure to acknowledge previous user requests a...,[If a user mentions a specific service or pric...,[Test if the assistant can recall and respond ...
2,3,User Refusal Not Acknowledged,The assistant fails to recognize and respect u...,[Implement a context-checking mechanism to rec...,Always acknowledge user refusals and adjust th...,"[If a user explicitly declines an offer, do no...",[Test conversations where users refuse offers ...
3,2,Context Awareness and Intent Recognition Issues,The assistant lacks the ability to retain and ...,[Enhance context retention capabilities to rem...,Ensure to retain context from previous user in...,[If a user specifies a time and number of peop...,[Test if the assistant can recall and use prev...
4,6,Misunderstanding User Intent for Ticket Orders,The assistant fails to recognize and respond a...,[Enhance context tracking to remember previous...,Ensure to clarify user intent when responses i...,[If the user indicates they are not ready to o...,[Test if the assistant correctly acknowledges ...
5,4,Reservation Request Handling Issues,The assistant lacks the capability to process ...,[Enhance the assistant's ability to recognize ...,"This assistant can help with various tasks, bu...","[If a user requests a reservation, respond wit...",[Test the assistant's response to multiple res...
6,5,Context Retention and Intent Misunderstanding,The assistant fails to retain user context and...,[Emphasize the importance of context retention...,Always retain and reference user preferences t...,"[If a user specifies a device for playback, co...",[Test conversations where users change playbac...
7,0,Appointment Scheduling Miscommunication,The assistant fails to retain context from pre...,[Enhance context retention capabilities to rem...,Ensure to retain context from previous message...,"[If a user specifies a date or time, confirm i...",[Test conversations where users change appoint...


In [11]:
repairs_parquet_path = DATA_PROCESSED / "uss_english_prompt_repairs.parquet"
df_repairs.to_parquet(repairs_parquet_path, index=False)
print("Saved prompt repairs parquet to:", repairs_parquet_path)

repairs_json_path = DATA_PROCESSED / "uss_english_prompt_repairs.json"
df_repairs.to_json(
    repairs_json_path,
    orient="records",
    indent=2,
    force_ascii=False
)
print("Saved prompt repairs JSON to:", repairs_json_path)


Saved prompt repairs parquet to: C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_prompt_repairs.parquet
Saved prompt repairs JSON to: C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_prompt_repairs.json


In [12]:
# Pretty print all prompt repairs (good for reviewing and for slides)
for _, row in df_repairs.sort_values("topic_id").iterrows():
    print("=" * 80)
    print(f"Topic {row['topic_id']}: {row['topic_label']}")
    print("\nRoot cause:")
    print("-", row["root_cause"])
    
    print("\nSuggested prompt changes:")
    for c in row["suggested_prompt_changes"]:
        print(" •", c)
    
    print("\nSystem prompt snippet:")
    print(row["system_prompt_snippet"])
    
    print("\nGuardrail rules:")
    for g in row["guardrail_rules"]:
        print(" •", g)
    
    print("\nEvaluation checks:")
    for e in row["evaluation_checks"]:
        print(" •", e)
    
    print("\n")


Topic 0: Appointment Scheduling Miscommunication

Root cause:
- The assistant fails to retain context from previous user inputs, leading to misunderstandings.

Suggested prompt changes:
 • Enhance context retention capabilities to remember previous user inputs.
 • Implement a confirmation step to clarify appointment details before finalizing.
 • Allow for flexible date and time adjustments based on user feedback.
 • Improve intent recognition for appointment-related queries.

System prompt snippet:
Ensure to retain context from previous messages and confirm user requests for appointments.

Guardrail rules:
 • If a user specifies a date or time, confirm it back to them before proceeding.
 • If the user expresses dissatisfaction with a proposed appointment, ask for clarification.
 • If the assistant does not recognize an intent, prompt the user for more details.

Evaluation checks:
 • Test conversations where users change appointment details to ensure the assistant acknowledges changes.
